# 🎬 Quality Validation, Ranking & Editorial Delivery Engine

Executes rigorous 4-step quality evaluation and editorial ranking over pooled candidates:
1. **Step 1 — Mechanical Gate**: Verifies AST IDs, monotonicity, strict 20.00s–60.00s duration, and <=3.00s hook offset.
2. **Step 2 — High-Overlap Deduplication**: Pairwise containment test (>=85% or >=60% with shared climax cut).
3. **Step 3 — Deterministic Rubric Sort**: Lexicographic key prioritizing floor compliance, cold open clarity, and hook strength.
4. **Step 4 — Cold-First Quality Critic**: LLM evaluates isolated transcript first to eliminate curse-of-knowledge bias.


## 1. Setup & Strongly-Typed Pydantic Schemas
Defines the concrete data model for final promos conforming to Page 14 of the specification.


In [1]:
import os
import sys
import re
import json
import time
from pathlib import Path
from typing import List, Optional, Literal, Dict, Any, Tuple, Union
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

# Paths
WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == "validator_ranker" else Path.cwd()
VALIDATOR_DIR = WORKSPACE_DIR / "validator_ranker"
VALIDATOR_DIR.mkdir(parents=True, exist_ok=True)

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
if str(VALIDATOR_DIR) not in sys.path:
    sys.path.insert(0, str(VALIDATOR_DIR))

# Pydantic Schemas
class TrimRecommendation(BaseModel):
    exact_opening_dialogue: str
    exact_ending_dialogue: str
    verbatim_trim_quote: Optional[str] = None
    boundary_type: Literal["element_boundary", "internal_cut"] = "element_boundary"
    requires_manual_editor_verification: bool = False

class PromoSegment(BaseModel):
    segment_index: int = Field(ge=1, le=3)
    episode: int
    scene: int
    start_element_id: str
    end_element_id: str
    start_timestamp: str
    end_timestamp: str
    duration_seconds: float
    duration_cs: int
    trim_quote_hint: Optional[str] = None
    segment_summary: str
    trim_recommendation: Optional[TrimRecommendation] = None

class RubricScores(BaseModel):
    standalone_coherence: int = Field(ge=1, le=5)
    hook_strength: int = Field(ge=1, le=5)
    cliffhanger_strength: int = Field(ge=1, le=5)
    emotional_impact: int = Field(ge=1, le=5)
    total_score: int = Field(ge=4, le=20)

class EditorialRationale(BaseModel):
    hook_explanation: str
    cliffhanger_explanation: str
    standalone_coherence_explanation: str

class CriticReview(BaseModel):
    critic_status: Literal["APPROVE", "FLAG_WITH_OBJECTION", "DEMOTE", "UNREVIEWED_BACKFILL"]
    makes_sense_standalone: bool
    compelling_within_3s: bool
    leaves_unresolved_cliffhanger: bool
    spoiler_free: bool
    continuity_preserved: bool
    editorial_notes: str

class PromoCandidate(BaseModel):
    promo_id: str
    rank: int
    composition_mode: Literal["single_episode", "multi_episode"]
    archetype: Literal["immediate_shock_opener", "visual_anomaly_mystery_escalation", "emotional_reversal_turning_point"]
    relationship_type: Optional[Literal["explicit", "implied", "thematic", "intra_episode"]] = None
    total_duration_seconds: float
    total_duration_cs: int
    hook_timestamp_offset_seconds: float
    segments: List[PromoSegment]
    rubric_scores: RubricScores
    editorial_rationale: EditorialRationale
    critic_review: CriticReview

print("[OK] Validation & Ranking Pydantic Schemas loaded.")


[OK] Validation & Ranking Pydantic Schemas loaded.


## 2. Initialize ValidatorRankerEngine
Loads canonical AST (`01_structured_episodes.json`), single-episode candidates (`02_batch_extractions.json`), and synthesized multi-episode candidates (`03_synthesized_candidates.json`).


In [2]:
try: 
    from validator_ranker import ValidatorRankerEngine
except (ImportError, ModuleNotFoundError):
        from validator_ranker.validator_ranker import ValidatorRankerEngine
ast_paths = [
    WORKSPACE_DIR / "ast_builder" / "01_structured_episodes.json",
    WORKSPACE_DIR / "01_structured_episodes.json",
]
ast_path = next((p for p in ast_paths if p.exists()), ast_paths[0])

extractions_paths = [
    WORKSPACE_DIR / "candidate_extractor" / "02_batch_extractions.json",
    WORKSPACE_DIR / "02_batch_extractions.json",
]
extractions_path = next((p for p in extractions_paths if p.exists()), extractions_paths[0])

synthesized_paths = [
    WORKSPACE_DIR / "cross_episode_synthesis" / "03_synthesized_candidates.json",
    WORKSPACE_DIR / "03_synthesized_candidates.json",
]
synthesized_path = next((p for p in synthesized_paths if p.exists()), synthesized_paths[0])

engine = ValidatorRankerEngine(
    ast_path=ast_path,
    extractions_path=extractions_path,
    synthesized_path=synthesized_path,
    final_promos_path=VALIDATOR_DIR / "final_promos.json",
    editorial_report_path=VALIDATOR_DIR / "editorial_report.md",
    execution_trace_path=VALIDATOR_DIR / "execution_trace.json",
)

print(f"AST Total Episodes Loaded: {len(engine.ast_data.get('episodes', []))}")
print(f"AST Indexed Elements    : {len(engine.element_lookup)}")
print(f"Single Candidates Input : {len(engine.extractions_data.get('candidates', []))}")
multi_count = len(engine.synthesized_data.get('synthesized_candidates') or engine.synthesized_data.get('candidates') or [])
print(f"Multi Candidates Input  : {multi_count}")


AST Total Episodes Loaded: 7
AST Indexed Elements    : 1526
Single Candidates Input : 10
Multi Candidates Input  : 3


## Mechanical Gate
- Pools all single- and multi-episode candidates.
- Checks AST element existence and start < end.
- Duration gate: strictly $2,000\text{ cs} \le \text{duration\_cs} \le 6,000\text{ cs}$ ($20.00\text{s} - 60.00\text{s}$).
- Hook offset gate: $0.0\text{s} \le \text{hook\_offset} \le 3.0\text{s}$, opening hook initiates within $300\text{ cs}$.
- Invalid durations are rejected outright and recorded in `execution_trace.json`.


In [3]:
gated_candidates = engine.step1_mechanical_gate()

df_gated = pd.DataFrame([{
    "ID": c["normalized_id"],
    "Mode": c["composition_mode"],
    "Duration (s)": c["calculated_duration_seconds"],
    "Duration (cs)": c["calculated_duration_cs"],
    "Hook Offset (s)": c["hook_timestamp_offset_seconds"],
    "Archetype": c["archetype"],
    "Total Score": c["total_rubric_score"]
} for c in gated_candidates])

print(f"\nSurviving Candidates after Mechanical Gate: {len(gated_candidates)}")
display(df_gated.head(10))



[STEP 1] EXECUTING MECHANICAL INVARIANT GATE
Total Raw Candidates Pooled: 13 (10 single, 3 multi)
  [REJECT] EP01_EL044_EP01_EL050 (single_episode): Duration 6930 cs (69.30s) outside 20.00s-60.00s gate

[STEP 1 RESULT] Passed Mechanical Gate: 12/13 (1 rejected)

Surviving Candidates after Mechanical Gate: 12


,ID,Mode,Duration (s),Duration (cs),Hook Offset (s),Archetype,Total Score
0,EP02_EL086_EP02_EL090,single_episode,34.8,3480,0.5,immediate_shock_opener,19
1,EP03_EL161_EP03_EL166,single_episode,50.9,5090,2.0,immediate_shock_opener,20
2,EP04_EL126_EP04_EL131,single_episode,35.7,3570,1.0,emotional_reversal_turning_point,19
3,EP05_EL087_EP05_EL095,single_episode,46.6,4660,1.5,emotional_reversal_turning_point,20
4,EP05_EL215_EP05_EL225,single_episode,53.6,5360,2.0,visual_anomaly_mystery_escalation,19
5,EP06_EL005_EP06_EL012,single_episode,34.2,3420,0.0,immediate_shock_opener,20
6,EP06_EL129_EP06_EL137,single_episode,50.2,5020,0.0,immediate_shock_opener,20
7,EP07_EL110_EP07_EL114,single_episode,37.5,3750,0.0,immediate_shock_opener,20
8,EP07_EL164_EP07_EL168,single_episode,27.1,2710,0.0,immediate_shock_opener,20
9,SYNTH_MULTI_01,multi_episode,51.9,5190,0.3,visual_anomaly_mystery_escalation,19


## 4. Step 2 — High-Overlap Deduplication 
- Same-episode candidates are compared pairwise.
- Drops lower-scoring clip if:
  $$\text{containment} = \frac{\text{intersection\_cs}}{\min(\text{dur\_a\_cs}, \text{dur\_b\_cs})} \ge 0.85$$
  **OR** if overlap $\ge 0.60$ and both candidates share the exact same `end_element_id` (shared climax).


In [4]:
deduped_candidates = engine.step2_high_overlap_deduplication(gated_candidates)

dropped_log = engine.trace_log["pipeline_stages"]["step2_deduplication"]["dropped"]
print(f"Surviving Candidates after Deduplication: {len(deduped_candidates)}")
print(f"Dropped Candidates ({len(dropped_log)}):")
for d in dropped_log:
    print(f"  - Dropped {d['dropped_id']} (Ep {d['episode']:02d}): {d['reason']}")



[STEP 2] EXECUTING HIGH-OVERLAP DEDUPLICATION

[STEP 2 RESULT] Deduplicated Pool: 12 (0 duplicate candidates dropped)
Surviving Candidates after Deduplication: 12
Dropped Candidates (0):


## Deterministic Rubric Sort
Sorts all surviving candidates using the exact lexicographic tuple key:
$$\text{rubric\_sort\_key} = (\text{passes\_min\_floor}, \text{standalone\_coherence}, \text{hook\_strength}, \text{cliffhanger\_strength}, \text{emotional\_impact}, \text{total\_rubric\_score})$$
where $\text{passes\_min\_floor} = (\text{standalone} \ge 4 \land \text{hook} \ge 3 \land \text{cliffhanger} \ge 3)$.  
This ensures high coherence and hook take precedence over raw sums.


In [5]:
sorted_candidates = engine.step3_deterministic_rubric_sort(deduped_candidates)

df_sorted = pd.DataFrame([{
    "Rank": c["rubric_rank"],
    "ID": c["normalized_id"],
    "Mode": c["composition_mode"],
    "Floor": c["passes_min_floor"],
    "Coherence": c["standalone_coherence"],
    "Hook": c["hook_strength"],
    "Cliffhanger": c["cliffhanger_strength"],
    "Emotion": c["emotional_impact"],
    "Score": c["total_rubric_score"]
} for c in sorted_candidates])

display(df_sorted.head(15))



[STEP 3] EXECUTING DETERMINISTIC RUBRIC SORT
Top 10 Rubric-Sorted Candidates:
  Rank 01 | EP03_EL161_EP03_EL166 | Mode: single_episode | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 20/20
  Rank 02 | EP05_EL087_EP05_EL095 | Mode: single_episode | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 20/20
  Rank 03 | EP06_EL005_EP06_EL012 | Mode: single_episode | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 20/20
  Rank 04 | EP06_EL129_EP06_EL137 | Mode: single_episode | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 20/20
  Rank 05 | EP07_EL110_EP07_EL114 | Mode: single_episode | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 20/20
  Rank 06 | EP07_EL164_EP07_EL168 | Mode: single_episode | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 20/20
  Rank 07 | SYNTH_MULTI_01  | Mode: multi_episode  | Floor: True  | Coherence: 5 | Hook: 5 | Cliff: 5 | Score: 19/20
  Rank 08 | SYNTH_MULTI_02  | Mode: multi_episode  | Floor: True  

,Rank,ID,Mode,Floor,Coherence,Hook,Cliffhanger,Emotion,Score
0,1,EP03_EL161_EP03_EL166,single_episode,True,5,5,5,5,20
1,2,EP05_EL087_EP05_EL095,single_episode,True,5,5,5,5,20
2,3,EP06_EL005_EP06_EL012,single_episode,True,5,5,5,5,20
3,4,EP06_EL129_EP06_EL137,single_episode,True,5,5,5,5,20
4,5,EP07_EL110_EP07_EL114,single_episode,True,5,5,5,5,20
5,6,EP07_EL164_EP07_EL168,single_episode,True,5,5,5,5,20
6,7,SYNTH_MULTI_01,multi_episode,True,5,5,5,4,19
7,8,SYNTH_MULTI_02,multi_episode,True,5,5,5,4,19
8,9,SYNTH_MULTI_03,multi_episode,True,5,5,5,4,19
9,10,EP02_EL086_EP02_EL090,single_episode,True,5,5,4,5,19


## 6. Step 4 — Cold-First Quality Critic 
- Takes a **20-candidate buffer** so that replacements are already Critic-approved if demotions occur.
- **Cold-First Prompt Structure**:
  - **(a) Isolated clip transcript first**
  - **(b) $\pm 3$ surrounding screenplay elements and episode logline second** to eliminate curse of knowledge.
- Evaluates the 5 editorial questions:
  1. `Cold Comprehension`: Does it make sense standalone?
  2. `3-Second Opening`: Is it compelling within $\le 3.0$s?
  3. `Unresolved Ending`: Does it cut at peak unresolved tension?
  4. `Spoiler / Over-Reveal Guard`: Does it avoid premature spoilers?
  5. `Multi-Episode Continuity`: Is dialogue continuity preserved across episodes?
- Outputs `"APPROVE"`, `"FLAG_WITH_OBJECTION"`, or `"DEMOTE"`.


In [6]:
# Review 20-candidate buffer
has_key = bool(os.environ.get("GEMINI_API_KEY"))
print(f"GEMINI_API_KEY Available: {has_key}")

reviewed_candidates = engine.step4_cold_first_quality_critic(
    sorted_candidates, buffer_size=20, use_llm=has_key
)

df_critic = pd.DataFrame([{
    "ID": c["normalized_id"],
    "Status": c["critic_review"]["critic_status"],
    "Standalone": c["critic_review"]["makes_sense_standalone"],
    "3s Hook": c["critic_review"]["compelling_within_3s"],
    "Cliffhanger": c["critic_review"]["leaves_unresolved_cliffhanger"],
    "Spoiler Free": c["critic_review"]["spoiler_free"],
    "Notes": c["critic_review"]["editorial_notes"][:50] + "..."
} for c in reviewed_candidates[:20]])

display(df_critic)


GEMINI_API_KEY Available: False

[STEP 4] EXECUTING COLD-FIRST QUALITY CRITIC (Buffer Size = 20)
  Critic Verdict | EP03_EL161_EP03_EL166 | Status: APPROVE              | Approved: Strong immediate hook, clean cliffhanger cut, cold comp...
  Critic Verdict | EP05_EL087_EP05_EL095 | Status: APPROVE              | Approved: Strong immediate hook, clean cliffhanger cut, cold comp...
  Critic Verdict | EP06_EL005_EP06_EL012 | Status: APPROVE              | Approved: Strong immediate hook, clean cliffhanger cut, cold comp...
  Critic Verdict | EP06_EL129_EP06_EL137 | Status: APPROVE              | Approved: Strong immediate hook, clean cliffhanger cut, cold comp...
  Critic Verdict | EP07_EL110_EP07_EL114 | Status: APPROVE              | Approved: Strong immediate hook, clean cliffhanger cut, cold comp...
  Critic Verdict | EP07_EL164_EP07_EL168 | Status: APPROVE              | Approved: Strong immediate hook, clean cliffhanger cut, cold comp...
  Critic Verdict | SYNTH_MULTI_01  | Status: 

,ID,Status,Standalone,3s Hook,Cliffhanger,Spoiler Free,Notes
0,EP03_EL161_EP03_EL166,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
1,EP05_EL087_EP05_EL095,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
2,EP06_EL005_EP06_EL012,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
3,EP06_EL129_EP06_EL137,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
4,EP07_EL110_EP07_EL114,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
5,EP07_EL164_EP07_EL168,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
6,SYNTH_MULTI_01,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
7,SYNTH_MULTI_02,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
8,SYNTH_MULTI_03,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."
9,EP02_EL086_EP02_EL090,APPROVE,True,True,True,True,"Approved: Strong immediate hook, clean cliffha..."


## 7. Step 5 — Demotion & Explicit Backfill
- Demotes/removes any `"DEMOTE"` or `"REJECT"` candidates.
- Fills Top 15 from surviving candidates.
- If a replacement comes from the 20-candidate buffer, it retains its Critic approval.
- If backfill reaches rank 21+ (unreviewed pool), it is explicitly labeled:
  `"critic_status": "UNREVIEWED_BACKFILL"`.


In [7]:
final_promos = engine.step5_demotion_and_backfill(reviewed_candidates, target_final_count=15)

df_final = pd.DataFrame([{
    "Rank": p.rank,
    "Promo ID": p.promo_id,
    "Mode": p.composition_mode,
    "Archetype": p.archetype,
    "Duration (s)": f"{p.total_duration_seconds:.2f}s",
    "Hook Offset": f"{p.hook_timestamp_offset_seconds:.2f}s",
    "Score": f"{p.rubric_scores.total_score}/20",
    "Critic Status": p.critic_review.critic_status
} for p in final_promos])

print("=" * 80)
print("FINAL TOP 15 PROMOTIONAL RECOMMENDATIONS:")
print("=" * 80)
display(df_final)



[STEP 5] EXECUTING DEMOTION & EXPLICIT BACKFILL (Target = 15)
  [FINAL]    Rank 01 | EP03_EL161_EP03_EL166 | Status: APPROVE
  [FINAL]    Rank 02 | EP05_EL087_EP05_EL095 | Status: APPROVE
  [FINAL]    Rank 03 | EP06_EL005_EP06_EL012 | Status: APPROVE
  [FINAL]    Rank 04 | EP06_EL129_EP06_EL137 | Status: APPROVE
  [FINAL]    Rank 05 | EP07_EL110_EP07_EL114 | Status: APPROVE
  [FINAL]    Rank 06 | EP07_EL164_EP07_EL168 | Status: APPROVE
  [FINAL]    Rank 07 | SYNTH_MULTI_01 | Status: APPROVE
  [FINAL]    Rank 08 | SYNTH_MULTI_02 | Status: APPROVE
  [FINAL]    Rank 09 | SYNTH_MULTI_03 | Status: APPROVE
  [FINAL]    Rank 10 | EP02_EL086_EP02_EL090 | Status: APPROVE
  [FINAL]    Rank 11 | EP04_EL126_EP04_EL131 | Status: APPROVE
  [FINAL]    Rank 12 | EP05_EL215_EP05_EL225 | Status: APPROVE

[STEP 5 RESULT] Emitted 12 Final Promos (Demotions: 0, Unreviewed Backfill: 0)
FINAL TOP 15 PROMOTIONAL RECOMMENDATIONS:


,Rank,Promo ID,Mode,Archetype,Duration (s),Hook Offset,Score,Critic Status
0,1,PROMO_001,single_episode,immediate_shock_opener,50.90s,2.00s,20/20,APPROVE
1,2,PROMO_002,single_episode,emotional_reversal_turning_point,46.60s,1.50s,20/20,APPROVE
2,3,PROMO_003,single_episode,immediate_shock_opener,34.20s,0.00s,20/20,APPROVE
3,4,PROMO_004,single_episode,immediate_shock_opener,50.20s,0.00s,20/20,APPROVE
4,5,PROMO_005,single_episode,immediate_shock_opener,37.50s,0.00s,20/20,APPROVE
5,6,PROMO_006,single_episode,immediate_shock_opener,27.10s,0.00s,20/20,APPROVE
6,7,PROMO_007,multi_episode,visual_anomaly_mystery_escalation,51.90s,0.30s,19/20,APPROVE
7,8,PROMO_008,multi_episode,immediate_shock_opener,59.20s,0.30s,19/20,APPROVE
8,9,PROMO_009,multi_episode,emotional_reversal_turning_point,57.10s,0.30s,19/20,APPROVE
9,10,PROMO_010,single_episode,immediate_shock_opener,34.80s,0.50s,19/20,APPROVE


## 8. Deep Dive: Promo Blueprints & Editing Trims
Inspects the concrete segment blueprints, verbatim opening/ending dialogue, internal trim quotes, and editor waveform check flags for video editors.


In [8]:
for p in final_promos[:3]:
    print("=" * 75)
    print(f"RANK {p.rank}: {p.promo_id} ({p.composition_mode.upper()}) | Archetype: {p.archetype}")
    print(f"Total Duration: {p.total_duration_seconds:.2f}s | Hook Offset: {p.hook_timestamp_offset_seconds:.2f}s | Score: {p.rubric_scores.total_score}/20")
    print(f"Hook Explanation       : {p.editorial_rationale.hook_explanation}")
    print(f"Cliffhanger Explanation: {p.editorial_rationale.cliffhanger_explanation}")
    print(f"Critic Status          : {p.critic_review.critic_status}")
    print(f"Critic Assessment      : {p.critic_review.editorial_notes}")
    print("-" * 75)
    print("CONSTITUENT SEGMENT DETAILS:")
    for seg in p.segments:
        trim = seg.trim_recommendation
        open_dlg = trim.exact_opening_dialogue if trim else 'N/A'
        end_dlg = trim.exact_ending_dialogue if trim else 'N/A'
        trim_q = trim.verbatim_trim_quote if (trim and trim.verbatim_trim_quote) else 'None'
        wf_chk = trim.requires_manual_editor_verification if trim else False
        print(f"  Segment {seg.segment_index}: Episode {seg.episode:02d} | [{seg.start_element_id} -> {seg.end_element_id}] ({seg.duration_seconds:.2f}s)")
        print(f"    Opening Line  : '{open_dlg}'")
        print(f"    Ending Line   : '{end_dlg}'")
        print(f"    Trim Quote    : '{trim_q}'")
        print(f"    Waveform Check: {wf_chk}")
    print()


RANK 1: PROMO_001 (SINGLE_EPISODE) | Archetype: immediate_shock_opener
Total Duration: 50.90s | Hook Offset: 2.00s | Score: 20/20
Hook Explanation       : Batman bursts into the penthouse confrontation with 'Then you're going to love me' and takes down the Joker's thugs with brutal CQC.
Cliffhanger Explanation: The Joker dangles Rachel Dawes out of a shattered skyscraper window and drops her into freefall, with Batman diving off the ledge after her.
Critic Status          : APPROVE
Critic Assessment      : Approved: Strong immediate hook, clean cliffhanger cut, cold comprehension validated.
---------------------------------------------------------------------------
CONSTITUENT SEGMENT DETAILS:
  Segment 1: Episode 03 | [EP03_EL161 -> EP03_EL166] (50.90s)
    Opening Line  : 'A little fight in you. I like that.'
    Ending Line   : 'Very poor choice of words...'
    Trim Quote    : 'None'
    Waveform Check: False

RANK 2: PROMO_002 (SINGLE_EPISODE) | Archetype: emotional_reversal_turni

## 9. Deliverables Export & Mechanical Summary
Emits `final_promos.json`, `editorial_report.md`, and `execution_trace.json`.  
Mirrors all output artifacts to project root for seamless delivery.


In [9]:
engine.export_deliverables(final_promos)

out_json = VALIDATOR_DIR / "final_promos.json"
out_report = VALIDATOR_DIR / "editorial_report.md"
out_trace = VALIDATOR_DIR / "execution_trace.json"

print("=" * 80)
print("VALIDATION & EDITORIAL DELIVERABLES AUDIT:")
print("=" * 80)
print(f"final_promos.json     : {out_json.resolve()} ({out_json.stat().st_size / 1024:.1f} KB)")
print(f"editorial_report.md   : {out_report.resolve()} ({out_report.stat().st_size / 1024:.1f} KB)")
print(f"execution_trace.json  : {out_trace.resolve()} ({out_trace.stat().st_size / 1024:.1f} KB)")
print(f"Total Promos Emitted  : {len(final_promos)}")
print("=" * 80)


[EXPORT] Wrote final_promos.json (12 promos)
[EXPORT] Wrote execution_trace.json
[EXPORT] Wrote editorial_report.md
VALIDATION & EDITORIAL DELIVERABLES AUDIT:
final_promos.json     : C:\Users\Anirudh Singh\OneDrive\Documents\Projects\Kuku FM\validator_ranker\final_promos.json (31.0 KB)
editorial_report.md   : C:\Users\Anirudh Singh\OneDrive\Documents\Projects\Kuku FM\validator_ranker\editorial_report.md (5.5 KB)
execution_trace.json  : C:\Users\Anirudh Singh\OneDrive\Documents\Projects\Kuku FM\validator_ranker\execution_trace.json (44.9 KB)
Total Promos Emitted  : 12
